In [1]:
import pandas as pd

# =========================
# Configuration
# =========================
INPUT_CSV = "raw_data.csv"

COL_MAP = "Map"
COL_ALG = "Algorithm"
COL_P   = "Desired Safe prob"
COL_RT  = "Runtime"

COL_REPLAN     = "Number Of Replans"
COL_ONLINE_SST = "Online Sum of Service Time"

# =========================
# Load data
# =========================
df = pd.read_csv(INPUT_CSV, low_memory=False)

def is_valid_runtime(x) -> bool:
    if pd.isna(x):
        return False
    s = str(x).strip()
    return s != "" and s.upper() != "NONE"

df["_valid_runtime"] = df[COL_RT].apply(is_valid_runtime)
df["_p_num"] = pd.to_numeric(df[COL_P], errors="coerce")

def build_algo_label(row) -> str:
    alg = str(row[COL_ALG]).strip()
    p_txt = str(row[COL_P]).strip()
    if alg == "CBSSsst":
        return "CBSSsst"
    return f"{alg} p={p_txt}"

df["_algo_label"] = df.apply(build_algo_label, axis=1)

GROUP_ORDER = {"CBSSsst": 0, "RCbssTS": 1, "RCbssTA": 2}
df["_group"] = df[COL_ALG].astype(str).str.strip().map(GROUP_ORDER).fillna(99).astype(int)

df["_p_for_sort"] = df["_p_num"]
df.loc[df["_group"] == 0, "_p_for_sort"] = float("inf")

# =========================
# Derive co-solved set from raw_data directly.
# Instance key excludes Delay prob (Planning) because CBSSsst uses 0.0
# while the robust algorithms use 0.25/0.5 — they run on the same underlying
# problem instance regardless of planning delay.
# Co-solved = every row sharing the same instance key has a valid Online SST.
# =========================
RUN_KEY = ["Map", "Number of agents", "Number of goals", "Instance", "Delay prob (Execution)"]

df["_solved"] = df[COL_ONLINE_SST].notna()
df_rep = df[df.groupby(RUN_KEY)["_solved"].transform("all")].copy()

# _p_num and _algo_label are already inherited from df
df_rep["_replan_num"]  = pd.to_numeric(df_rep[COL_REPLAN], errors="coerce").fillna(0)
df_rep["_online_sst"]  = pd.to_numeric(df_rep[COL_ONLINE_SST], errors="coerce")
df_rep["_runtime_num"] = pd.to_numeric(df_rep[COL_RT], errors="coerce")

rep_summary = (
    df_rep.groupby([COL_MAP, "_algo_label"], dropna=False)
          .agg(
              replan_ge_1_count=("_replan_num", lambda s: int((s >= 1).sum())),
              online_sst_mean=("_online_sst", "mean"),
              avg_runtime_mean=("_runtime_num", "mean"),
          )
          .reset_index()
)

# =========================
# Print table per map
# =========================
for map_name, g in df.groupby(COL_MAP, dropna=False):
    summary = (
        g.groupby(["_algo_label", "_group", "_p_for_sort"], dropna=False)
         .agg(total=(COL_RT, "size"), success=("_valid_runtime", "sum"))
         .reset_index()
    )

    summary["Success rate"] = (summary["success"] / summary["total"] * 100).round(2)

    rep_map = rep_summary[rep_summary[COL_MAP].astype(str) == str(map_name)]
    summary = summary.merge(
        rep_map[[COL_MAP, "_algo_label", "replan_ge_1_count", "online_sst_mean", "avg_runtime_mean"]],
        how="left",
        on=["_algo_label"],
    )

    summary["replan_ge_1_count"] = summary["replan_ge_1_count"].fillna(0).astype(int)
    summary["online_sst_mean"]   = summary["online_sst_mean"].round(2)
    summary["avg_runtime_mean"]  = summary["avg_runtime_mean"].round(2)

    summary = summary.sort_values(
        by=["_group", "_p_for_sort", "_algo_label"],
        ascending=[True, True, True],
        na_position="last"
    ).reset_index(drop=True)

    col1 = "planner configuration"
    rows = [
        (
            r["_algo_label"],
            f"{r['Success rate']:.2f}",
            int(r["_group"]),
            int(r["replan_ge_1_count"]),
            "NA" if pd.isna(r["avg_runtime_mean"]) else f"{r['avg_runtime_mean']:.2f}",
            "NA" if pd.isna(r["online_sst_mean"])   else f"{r['online_sst_mean']:.2f}",
        )
        for _, r in summary.iterrows()
    ]

    width = max(len(col1), max((len(name) for name, *_ in rows), default=len(col1)))

    print("\n")
    print(f"Map: {map_name}")
    print("*" * 60)
    print(f"\n{col1.ljust(width)}  Success rate (%)  Replan>=1 (count)  Avg runtime (sec)  Avg online SST")
    print("=" * (width + 84))

    # Separator after the CBSSsst block (group 0) — derived dynamically
    # so it stays correct regardless of how many baseline rows exist.
    group0_cut = sum(1 for _, _, grp, *_ in rows if grp == 0)

    for i, (name, rate, grp, rep_cnt, rt_mean, sst_mean) in enumerate(rows):
        if i == group0_cut and group0_cut > 0 and i < len(rows):
            print("-" * (width + 84))
        if grp == 2 and i > 0 and rows[i - 1][2] != 2:
            print("-" * (width + 84))
        print(
            f"{name.ljust(width)}  {rate.rjust(6)}           "
            f"{str(rep_cnt).rjust(6)}           "
            f"{str(rt_mean).rjust(14)}           "
            f"{str(sst_mean).rjust(12)}"
        )




Map: maze-32-32-2
************************************************************

planner configuration  Success rate (%)  Replan>=1 (count)  Avg runtime (sec)  Avg online SST
CBSSsst                 83.67               33                    16.25                 181.47
---------------------------------------------------------------------------------------------------------
RCbssTS p=0.05          80.33               13                    15.62                 180.93
RCbssTS p=0.25          79.72               13                    15.66                 180.81
RCbssTS p=0.5           78.89               12                    15.64                 180.93
RCbssTS p=0.8           76.00                1                    15.48                 180.79
RCbssTS p=0.95          73.00                0                    15.59                 180.94
RCbssTS p=0.99          70.50                0                    15.61                 180.92
RCbssTS p=0.999         65.67                0       

In [2]:
import pandas as pd
from scipy.stats import binomtest
from IPython.display import display

# === Load data ===
df = pd.read_csv("raw_data.csv", low_memory=False)

# === Clean / define success ===
df["Desired Safe prob"] = pd.to_numeric(df["Desired Safe prob"], errors="coerce")
df["Success"] = df["Online Sum of Service Time"].notna()

map_col = "Map"
algorithms = ["RCbssTS", "RCbssTA"]

p_values = [0.05, 0.25, 0.5, 0.8, 0.95, 0.99, 0.999, 0.9999]

filtered = df[
    (df["Desired Safe prob"].isin(p_values)) &
    (df["Algorithm"].isin(algorithms))
].copy()


# === Columns that identify the same instance across algorithms ===
key_cols = [
    c for c in [
        "Map", "Number of agents", "Number of goals",
        "Desired Safe prob", "Instance", "Delay prob (Execution)",
    ]
    if c in filtered.columns
]


def exact_mcnemar_pvalue(group_df):
    paired = group_df.pivot_table(
        index=key_cols, columns="Algorithm", values="Success", aggfunc="first"
    )

    if not {"RCbssTA", "RCbssTS"}.issubset(paired.columns):
        return float("nan")

    paired = paired.dropna(subset=["RCbssTA", "RCbssTS"])
    if paired.empty:
        return float("nan")

    ta = paired["RCbssTA"].astype(bool)
    ts = paired["RCbssTS"].astype(bool)
    b = (ta & ~ts).sum()
    c = (~ta & ts).sum()
    n_discordant = b + c

    if n_discordant == 0:
        return 1.0

    return binomtest(k=min(b, c), n=n_discordant, p=0.5, alternative="two-sided").pvalue


def build_success_table(group_col):
    summary = (
        filtered
        .groupby([map_col, group_col, "Algorithm"], as_index=False)
        .agg(SuccessRate=("Success", "mean"))
    )
    summary["SuccessRate"] *= 100

    table = summary.pivot_table(
        index=[map_col, group_col], columns="Algorithm", values="SuccessRate"
    ).reset_index()
    table.columns.name = None

    pvals = pd.DataFrame(
        [
            (m, gval, exact_mcnemar_pvalue(gdf))
            for (m, gval), gdf in filtered.groupby([map_col, group_col])
        ],
        columns=[map_col, group_col, "McNemar p-value"],
    )

    table = table.merge(pvals, on=[map_col, group_col], how="left")
    table = table.rename(columns={
        map_col: "Map",
        "RCbssTS": "RCbssTS success (%)",
        "RCbssTA": "RCbssTA success (%)",
    })

    success_cols = [c for c in ["RCbssTS success (%)", "RCbssTA success (%)"] if c in table.columns]
    table[success_cols] = table[success_cols].round(2)
    table = table[["Map", group_col] + success_cols + ["McNemar p-value"]]
    table["Significant"] = table["McNemar p-value"].apply(lambda p: "yes" if pd.notna(p) and p < 0.05 else "no")
    table["McNemar p-value"] = table["McNemar p-value"].apply(
        lambda p: "<0.0001" if pd.notna(p) and p < 0.0001 else round(p, 4)
    )
    return table


by_goals  = build_success_table("Number of goals")
by_agents = build_success_table("Number of agents")

for map_name in sorted(filtered[map_col].unique()):
    print("\n" + "=" * 60)
    print(f"Map: {map_name}")
    print("=" * 60)

    print("\nSuccess rate by number of goals:")
    display(by_goals[by_goals["Map"] == map_name].sort_values("Number of goals").reset_index(drop=True))

    print("\nSuccess rate by number of agents:")
    display(by_agents[by_agents["Map"] == map_name].sort_values("Number of agents").reset_index(drop=True))



Map: maze-32-32-2

Success rate by number of goals:


,Map,Number of goals,RCbssTS success (%),RCbssTA success (%),McNemar p-value,Significant
0,maze-32-32-2,15,92.90,97.73,<0.0001,yes
1,maze-32-32-2,20,85.29,96.42,<0.0001,yes
2,maze-32-32-2,25,41.25,94.31,<0.0001,yes



Success rate by number of agents:


,Map,Number of agents,RCbssTS success (%),RCbssTA success (%),McNemar p-value,Significant
0,maze-32-32-2,20,68.25,92.47,<0.0001,yes
1,maze-32-32-2,30,75.56,95.97,<0.0001,yes
2,maze-32-32-2,40,77.67,98.97,<0.0001,yes
3,maze-32-32-2,50,71.11,97.19,<0.0001,yes



Map: random-32-32-20

Success rate by number of goals:


...

,Map,Number of agents,RCbssTS success (%),RCbssTA success (%),McNemar p-value,Significant
0,room-32-32-4,20,74.67,91.58,<0.0001,yes
1,room-32-32-4,30,75.03,91.86,<0.0001,yes
2,room-32-32-4,40,79.64,96.25,<0.0001,yes
3,room-32-32-4,50,73.67,95.83,<0.0001,yes



Map: warehouse-10-20-10-2-1

Success rate by number of goals:


,Map,Number of goals,RCbssTS success (%),RCbssTA success (%),McNemar p-value,Significant
0,warehouse-10-20-10-2-1,15,97.21,99.21,<0.0001,yes
1,warehouse-10-20-10-2-1,20,93.17,97.44,<0.0001,yes
2,warehouse-10-20-10-2-1,25,66.48,95.08,<0.0001,yes



Success rate by number of agents:


,Map,Number of agents,RCbssTS success (%),RCbssTA success (%),McNemar p-value,Significant
0,warehouse-10-20-10-2-1,20,88.31,96.78,<0.0001,yes
1,warehouse-10-20-10-2-1,30,85.50,97.67,<0.0001,yes
2,warehouse-10-20-10-2-1,40,86.81,97.22,<0.0001,yes
3,warehouse-10-20-10-2-1,50,81.86,97.31,<0.0001,yes


In [3]:
import numpy as np
from scipy.stats import ttest_rel
from IPython.display import display

# Reuses `filtered`, `map_col`, `key_cols` from cell-1.
# key_cols is the same pairing key — here used on Online SST (co-solved only).

SST_COL = "Online Sum of Service Time"
filtered[SST_COL] = pd.to_numeric(filtered[SST_COL], errors="coerce")


def paired_sst_ttest(group_df):
    piv = group_df.pivot_table(
        index=key_cols, columns="Algorithm", values=SST_COL, aggfunc="first"
    )

    if not {"RCbssTS", "RCbssTA"}.issubset(piv.columns):
        return np.nan, np.nan, 0, np.nan

    piv = piv.dropna(subset=["RCbssTS", "RCbssTA"])
    n = len(piv)
    if n == 0:
        return np.nan, np.nan, 0, np.nan

    ts_mean = piv["RCbssTS"].mean()
    ta_mean = piv["RCbssTA"].mean()
    pval = ttest_rel(piv["RCbssTS"], piv["RCbssTA"]).pvalue if n >= 2 else np.nan
    return ts_mean, ta_mean, n, pval


def build_sst_table(group_col):
    rows = [
        (m, gval, *paired_sst_ttest(gdf))
        for (m, gval), gdf in filtered.groupby([map_col, group_col])
    ]

    table = pd.DataFrame(
        rows,
        columns=["Map", group_col, "RCbssTS SST (co-solved)", "RCbssTA SST (co-solved)", "Co-solved N", "t-test p-value"],
    )

    table[["RCbssTS SST (co-solved)", "RCbssTA SST (co-solved)"]] = (
        table[["RCbssTS SST (co-solved)", "RCbssTA SST (co-solved)"]].round(2)
    )
    table["Significant"] = table["t-test p-value"].apply(
        lambda p: "yes" if pd.notna(p) and p < 0.05 else "no"
    )
    table["t-test p-value"] = table["t-test p-value"].apply(
        lambda p: "<0.0001" if pd.notna(p) and p < 0.0001 else (round(p, 4) if pd.notna(p) else np.nan)
    )
    return table


sst_by_goals  = build_sst_table("Number of goals")
sst_by_agents = build_sst_table("Number of agents")

for map_name in sorted(filtered[map_col].unique()):
    print("\n" + "=" * 60)
    print(f"Map: {map_name}  |  Online SST on co-solved instances (paired t-test)")
    print("=" * 60)

    print("\nOnline SST by number of goals:")
    display(sst_by_goals[sst_by_goals["Map"] == map_name].sort_values("Number of goals").reset_index(drop=True))

    print("\nOnline SST by number of agents:")
    display(sst_by_agents[sst_by_agents["Map"] == map_name].sort_values("Number of agents").reset_index(drop=True))



Map: maze-32-32-2  |  Online SST on co-solved instances (paired t-test)

Online SST by number of goals:


,Map,Number of goals,RCbssTS SST (co-solved),RCbssTA SST (co-solved),Co-solved N,t-test p-value,Significant
0,maze-32-32-2,15,151.97,151.94,4453,0.0336,yes
1,maze-32-32-2,20,211.30,210.20,4064,<0.0001,yes
2,maze-32-32-2,25,312.99,330.64,1911,<0.0001,yes



Online SST by number of agents:


,Map,Number of agents,RCbssTS SST (co-solved),RCbssTA SST (co-solved),Co-solved N,t-test p-value,Significant
0,maze-32-32-2,20,300.94,305.27,2396,0.0002,yes
1,maze-32-32-2,30,219.30,221.77,2699,<0.0001,yes
2,maze-32-32-2,40,168.31,170.78,2781,<0.0001,yes
3,maze-32-32-2,50,138.14,140.21,2552,<0.0001,yes



Map: random-32-32-20  |  Online SST on co-solved instances (paired t-test)

Online SST by number of goals:


...

,Map,Number of agents,RCbssTS SST (co-solved),RCbssTA SST (co-solved),Co-solved N,t-test p-value,Significant
0,room-32-32-4,20,238.73,241.45,2636,<0.0001,yes
1,room-32-32-4,30,186.70,186.59,2660,0.7481,no
2,room-32-32-4,40,157.21,157.88,2841,0.0096,yes
3,room-32-32-4,50,129.89,131.43,2633,<0.0001,yes



Map: warehouse-10-20-10-2-1  |  Online SST on co-solved instances (paired t-test)

Online SST by number of goals:


,Map,Number of goals,RCbssTS SST (co-solved),RCbssTA SST (co-solved),Co-solved N,t-test p-value,Significant
0,warehouse-10-20-10-2-1,15,335.40,335.42,4659,0.2169,no
1,warehouse-10-20-10-2-1,20,476.73,479.53,4445,<0.0001,yes
2,warehouse-10-20-10-2-1,25,740.96,781.16,3161,<0.0001,yes



Online SST by number of agents:


,Map,Number of agents,RCbssTS SST (co-solved),RCbssTA SST (co-solved),Co-solved N,t-test p-value,Significant
0,warehouse-10-20-10-2-1,20,658.51,679.65,3169,<0.0001,yes
1,warehouse-10-20-10-2-1,30,502.07,510.11,3055,<0.0001,yes
2,warehouse-10-20-10-2-1,40,422.17,430.62,3110,<0.0001,yes
3,warehouse-10-20-10-2-1,50,371.99,379.42,2931,0.0001,yes
